# KNN Purchase Probability

여기서는 전처리 노트북에서 만들어 둔 KNN용 데이터셋을 가지고, 온라인 쇼핑 방문자가 실제로 구매할 가능성이 얼마나 되는지 예측해본다.

KNN은 직관이 꽤 단순하다. 새 방문자가 들어왔을 때, 과거 데이터에서 행동 패턴이 비슷한 방문자들을 찾고, 그 사람들이 실제로 구매했는지를 보는 방식이다. 그래서 이 모델은 단순히 점수만 내는 것이 아니라, “어떤 방문자들이 구매 가능성이 높게 묶이는지”까지 같이 살펴볼 수 있다.

진행 순서는 간단하다. 먼저 기준 모델로 KNN이 어떤 식으로 확률을 내는지 확인하고, 그 다음 K값, 거리 기준, 가중치 방식을 바꿔가며 더 나은 설정을 찾는다. 마지막으로 테스트 데이터에서 최종 성능을 확인하고, 예측 상위 그룹이 어떤 행동 특징을 보이는지 분석한다.

## 0. KNN 최종 산출물 로드

In [97]:
from pathlib import Path

import pandas as pd

# KNN 전처리 산출물이 저장된 csv 폴더 경로 설정
DATA_DIR = Path("csv")

# 전처리 노트북에서 저장한 KNN 학습/테스트 데이터를 불러옴
X_train_knn = pd.read_csv(DATA_DIR / "X_train_knn.csv")
X_test_knn = pd.read_csv(DATA_DIR / "X_test_knn.csv")
y_train_knn = pd.read_csv(DATA_DIR / "y_train_knn.csv")["Revenue"]
y_test_knn = pd.read_csv(DATA_DIR / "y_test_knn.csv")["Revenue"]

# 데이터 크기와 구매 비율을 확인해 정상적으로 로드되었는지 점검
print(f"X_train_knn: {X_train_knn.shape}")
print(f"X_test_knn : {X_test_knn.shape}")
print(f"y_train_knn: {y_train_knn.shape}, positive_ratio = {y_train_knn.mean():.4f}")
print(f"y_test_knn : {y_test_knn.shape}, positive_ratio = {y_test_knn.mean():.4f}")

X_train_knn: (16676, 11)
X_test_knn : (2466, 11)
y_train_knn: (16676,), positive_ratio = 0.5000
y_test_knn : (2466,), positive_ratio = 0.1549


## 1. 기준 KNN 모델

먼저 가장 단순한 기준 모델을 하나 만들어본다. 여기서 `K=5`는 최종 선택값이라기보다는, KNN이 예측 확률을 어떤 식으로 만들어내는지 확인하기 위한 출발점이다.

전처리 과정에서 학습 데이터에는 SMOTE가 적용되어 구매/미구매 비율이 50:50에 가깝게 맞춰져 있다. 그래서 이 기준 모델의 평균 예측 확률은 실제 테스트셋 구매율인 15.5%보다 높게 나올 수 있다. 실제로 평균 예측 확률이 약 34.5%였고, 이를 K=5 기준으로 보면 가까운 5명 중 평균 약 1.7명이 구매자인 이웃으로 잡힌 셈이다.

In [98]:
from sklearn.neighbors import KNeighborsRegressor

# 기준 모델은 KNN의 예측값 형태를 확인하기 위한 단순 출발점
BASE_K = 5

# K=5 기준 KNN 회귀 모델 생성 및 학습
base = KNeighborsRegressor(n_neighbors = BASE_K)
base.fit(X_train_knn, y_train_knn)

# 테스트 데이터에 대해 구매 확률 점수 예측
base_proba = base.predict(X_test_knn)

# 예측 확률의 앞부분을 확인
print("\n샘플 10개")
print(base_proba[: 10])

# 예측 확률 분포를 간단히 요약
print("\n예측 확률 요약")
print(pd.Series(base_proba).describe())


샘플 10개
[0.  0.2 0.6 0.  0.  0.2 0.  0.8 0.8 0.4]

예측 확률 요약
count    2466.000000
mean        0.344850
std         0.361661
min         0.000000
25%         0.000000
50%         0.200000
75%         0.600000
max         1.000000
dtype: float64


## 2. K 후보 탐색

기준 모델에서는 `K=5`를 썼지만, 이 값이 가장 좋은 값이라고 볼 수는 없다. KNN에서 K는 “몇 명의 이웃을 볼 것인가”를 정하는 값이라서, 모델 성능에 직접적인 영향을 준다.

그래서 아래 후보들을 두고 비교해본다.

```python
K_CANDIDATES = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101, 151]
```

이 후보들은 그냥 아무 숫자나 고른 것이 아니다. 작은 K는 예측이 많이 흔들릴 수 있으니 촘촘하게 보고, 큰 K는 변화가 상대적으로 완만하니 조금 넓은 간격으로 확인한다.

| 구간 | 후보 | 보는 이유 |
|---|---|---|
| 작은 K | `1`, `3`, `5`, `7` | 가까운 몇 명만 보는 구간이다. 세부 패턴은 잘 잡지만 노이즈에도 민감하다. |
| 중간 K | `11`, `15`, `21`, `31` | 너무 예민한 상태에서 조금씩 안정화되는 구간이다. |
| 큰 K | `51`, `75`, `101`, `151` | 많은 이웃을 평균내는 구간이다. 안정적이지만 개별 패턴은 흐려질 수 있다. |

예를 들어 `K=1`이면 이웃 한 명이 예측 전체를 결정한다. 반면 `K=5`에서는 한 명의 영향이 20%로 줄어든다. 이런 이유로 작은 K는 조금 더 자세히 비교한다.

또 현재 SMOTE 이후 학습 데이터가 약 1.7만 개라서, `sqrt(N)` 경험칙을 생각하면 100명대 근처도 한 번은 확인해볼 만하다. 그래서 `101`, `151` 같은 큰 후보도 포함했다.

## 2-1. 직접 반복문으로 K 비교

먼저 직접 `for`문을 돌려서 K값별 성능을 확인한다. 이렇게 하면 GridSearchCV를 쓰기 전에, K가 커질수록 성능이 어떤 방향으로 움직이는지 감을 잡을 수 있다.

여기서는 `StratifiedKFold`를 사용한다. 구매자가 적은 데이터이기 때문에, 각 fold마다 구매/미구매 비율이 비슷하게 유지되도록 나누는 것이 중요하다.

평가 지표는 다음처럼 본다.

**ROC-AUC**는 구매자에게 비구매자보다 더 높은 점수를 주는지 본다. 즉, 구매 가능성이 높은 사람을 위쪽으로 잘 올리는지 확인하는 지표다.

**PR-AUC**는 구매자가 적은 상황에서 더 중요하다. 모델이 “구매 가능성이 높다”고 본 사람들 중 실제 구매자가 얼마나 잘 섞여 있는지 보는 데 도움이 된다.

**Brier Score**는 예측 확률 자체가 얼마나 믿을 만한지 본다. 예측 확률과 실제 결과의 차이를 보는 지표이고, 낮을수록 좋다.

**Precision**과 **Recall**은 나중에 임계값을 정해서 실제 의사결정에 연결할 때 본다. 이번 분석에서는 확률 점수 자체가 중요하므로 보조적으로 확인한다.

**F1 Score**는 Precision과 Recall을 하나로 묶어 보는 값이다. 다만 여기서는 0/1 분류 자체보다 구매 확률 점수를 만드는 것이 핵심이라 보조 지표로 둔다.

In [99]:
import numpy as np

from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# 비교할 K 후보 목록 설정
K_CANDIDATES = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101, 151]

# 구매/미구매 비율을 유지하면서 5-fold 교차검증을 수행
cv = StratifiedKFold(n_splits = 5, 
                     shuffle = True, 
                     random_state = 42)

# 각 K 후보의 교차검증 결과를 저장할 리스트
rows = []

for k in K_CANDIDATES:
    # 현재 K에서 fold별 평가 점수를 저장
    auc_scores = []
    ap_scores = []
    brier_scores = []

    for train_idx, valid_idx in cv.split(X_train_knn, y_train_knn):
        # 현재 fold의 학습/검증 데이터 분리
        X_tr = X_train_knn.iloc[train_idx]
        X_va = X_train_knn.iloc[valid_idx]
        y_tr = y_train_knn.iloc[train_idx]
        y_va = y_train_knn.iloc[valid_idx]

        # 현재 K값으로 KNN 모델 학습
        model = KNeighborsRegressor(n_neighbors = k)
        model.fit(X_tr, y_tr)

        # 검증 데이터에 대해 구매 확률 예측
        proba = model.predict(X_va)

        # fold별 평가 지표 저장
        auc_scores.append(roc_auc_score(y_va, proba))
        ap_scores.append(average_precision_score(y_va, proba))
        brier_scores.append(brier_score_loss(y_va, proba))

    # 현재 K의 fold 평균 성능을 저장
    rows.append({
        "K": k,
        "ROC_AUC": np.mean(auc_scores),
        "PR_AUC": np.mean(ap_scores),
        "Brier": np.mean(brier_scores),
    })

# k_result: K 후보별 5-fold 교차검증 평균 성능을 모은 표
# K       : 사용한 이웃 수
# ROC_AUC : 구매자를 비구매자보다 높은 점수로 정렬하는 능력
# PR_AUC  : 구매자가 적은 불균형 상황에서 구매자 탐지 품질
# Brier   : 예측 확률과 실제값의 차이, 낮을수록 좋음
k_result = pd.DataFrame(rows)
k_result = k_result.sort_values("ROC_AUC", ascending = False)

# K 후보별 성능 비교 결과 출력
print("직접 반복문으로 비교한 K 후보")
display(k_result)

직접 반복문으로 비교한 K 후보


,K,ROC_AUC,PR_AUC,Brier
1,3,0.893524,0.833284,0.134305
2,5,0.890562,0.838046,0.142768
3,7,0.884083,0.838814,0.148201
4,11,0.865600,0.826786,0.157450
0,1,0.864298,0.793811,0.135704
5,15,0.849931,0.814086,0.164103
6,21,0.832764,0.797788,0.171043
7,31,0.812244,0.774466,0.178682
8,51,0.787910,0.741030,0.187246
9,75,0.772618,0.722939,0.192373


이 데이터는 구매자가 적은 불균형 데이터이고, 우리는 단순 정답 맞히기보다 구매 가능성 점수를 만들고 싶다. 그래서 정확도 하나만 보기보다는 ROC-AUC, PR-AUC, Brier Score를 중심으로 보고, Precision과 Recall은 운영 관점에서 함께 참고한다.

### 2-2. GridSearchCV로 K 찾기

이번에는 같은 탐색을 `GridSearchCV`에 맡겨본다. 앞에서는 K값만 직접 비교했다면, 여기서는 거리 기준과 가중치 방식까지 바꿔가며 확인한다.

즉, “몇 명을 볼 것인가”뿐 아니라 “어떤 거리로 가까움을 볼 것인가”, “가까운 이웃을 더 중요하게 볼 것인가”까지 같이 비교한다.

In [100]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

# GridSearchCV가 각 후보 모델을 ROC-AUC 기준으로 평가하도록 scorer 설정
auc_scorer = make_scorer(roc_auc_score, response_method = "predict")

# p = 1: 맨해튼 거리, p = 2: 유클리드 거리
# weights = "uniform": 선택된 K개 이웃을 모두 같은 비중으로 반영
# weights = "distance": 가까운 이웃일수록 더 큰 비중으로 반영
knn_settings = [
    {
        "name": "Euclidean + uniform",
        "weights": "uniform",
        "p": 2,
    },
    {
        "name": "Euclidean + distance",
        "weights": "distance",
        "p": 2,
    },
    {
        "name": "Manhattan + uniform",
        "weights": "uniform",
        "p": 1,
    },
    {
        "name": "Manhattan + distance",
        "weights": "distance",
        "p": 1,
    },
]

# 설정별 GridSearchCV 결과를 저장할 딕셔너리
grid_results = {}

for setting in knn_settings:
    # 현재 거리 기준과 가중치 방식으로 K 후보 탐색 모델 구성
    grid = GridSearchCV(
        estimator = KNeighborsRegressor(
            weights = setting["weights"],
            p = setting["p"],
        ),
        param_grid = {"n_neighbors": K_CANDIDATES},
        scoring = auc_scorer,
        cv = cv,
    )

    # 현재 설정에서 K 후보별 교차검증 실행
    grid.fit(X_train_knn, y_train_knn)

    # result_df: 설정별 K 후보의 교차검증 ROC-AUC 평균과 표준편차를 모은 표
    result_df = pd.DataFrame(grid.cv_results_)
    result_df = result_df[["param_n_neighbors", "mean_test_score", "std_test_score"]]
    result_df = result_df.rename(columns = {
        "param_n_neighbors": "K",
        "mean_test_score": "CV_ROC_AUC",
        "std_test_score": "CV_ROC_AUC_std",
    })

    # ROC-AUC가 높은 K부터 확인할 수 있도록 정렬
    result_df = result_df.sort_values("CV_ROC_AUC", ascending = False)

    # 설정 이름을 key로 하여 결과표 저장
    grid_results[setting["name"]] = result_df

    # 현재 설정에서 선택된 K와 전체 K 후보 결과 출력
    print(f"\n[{setting['name']}]")
    print(f"선택된 K: {grid.best_params_['n_neighbors']}")
    display(result_df)


[Euclidean + uniform]
선택된 K: 3


,K,CV_ROC_AUC,CV_ROC_AUC_std
1,3,0.893524,0.002851
2,5,0.890562,0.006030
3,7,0.884083,0.006746
4,11,0.865600,0.006010
0,1,0.864298,0.003410
5,15,0.849931,0.004793
6,21,0.832764,0.004858
7,31,0.812244,0.004037
8,51,0.787910,0.004678
9,75,0.772618,0.005332



[Euclidean + distance]
선택된 K: 7


,K,CV_ROC_AUC,CV_ROC_AUC_std
3,7,0.918605,0.004252
4,11,0.915676,0.003813
2,5,0.913721,0.003997
5,15,0.909827,0.002350
1,3,0.903672,0.002410
6,21,0.901458,0.002535
7,31,0.889080,0.001976
8,51,0.873327,0.002757
0,1,0.864298,0.003410
9,75,0.861099,0.003748



[Manhattan + uniform]
선택된 K: 3


,K,CV_ROC_AUC,CV_ROC_AUC_std
1,3,0.898207,0.004346
2,5,0.895032,0.005602
3,7,0.887240,0.005931
4,11,0.870466,0.005514
0,1,0.868015,0.003534
5,15,0.855507,0.004877
6,21,0.838860,0.004819
7,31,0.817683,0.005615
8,51,0.793372,0.005315
9,75,0.776628,0.006633



[Manhattan + distance]
선택된 K: 7


,K,CV_ROC_AUC,CV_ROC_AUC_std
3,7,0.921990,0.004087
4,11,0.918310,0.003669
2,5,0.918216,0.003755
5,15,0.912868,0.003517
1,3,0.908941,0.003147
6,21,0.904614,0.003293
7,31,0.892937,0.003229
8,51,0.876788,0.003239
0,1,0.868015,0.003534
9,75,0.864664,0.004762


### 2-3. 결과 분석 및 K값 결정

4가지 설정을 비교해보니, 가장 좋은 결과는 **Manhattan + distance** 조합에서 나왔다.

| 설정 | 선택된 K | 최고 CV ROC-AUC | 해석 |
|---|---:|---:|---|
| Euclidean + uniform | 3 | 0.893524 | 기본 설정에 가까운 결과다. 작은 K에서 가장 좋았다. |
| Euclidean + distance | 7 | 0.918605 | 가까운 이웃을 더 크게 반영하자 성능이 꽤 좋아졌다. |
| Manhattan + uniform | 3 | 0.898207 | 유클리드 uniform보다 조금 높지만 차이는 크지 않다. |
| Manhattan + distance | 7 | **0.921990** | 전체 조합 중 가장 높은 ROC-AUC를 보였다. |

가장 눈에 띄는 부분은 `weights`다. `uniform`은 선택된 이웃들을 모두 똑같이 평균내지만, `distance`는 더 가까운 이웃에게 더 큰 비중을 준다. 이번 데이터에서는 이 차이가 꽤 크게 나타났다.

거리 기준에서는 `Manhattan(p = 1)`이 `Euclidean(p = 2)`보다 조금 더 좋았다. 쇼핑 행동 데이터에는 페이지 수나 체류 시간처럼 분포가 한쪽으로 긴 변수들이 있기 때문에, 큰 차이를 제곱해서 키우는 유클리드 거리보다 맨해튼 거리가 조금 더 안정적으로 작동한 것으로 볼 수 있다.

K값은 distance 가중치를 쓴 두 조합에서 모두 `K=7`이 선택되었다. 너무 적은 이웃만 보면 노이즈에 민감하고, 너무 많은 이웃을 보면 개별 패턴이 흐려진다. 이번 결과에서는 가까운 7명을 보되, 그중에서도 더 가까운 이웃을 더 중요하게 보는 방식이 가장 균형이 좋았다.

그래서 이후 최종 모델은 아래 설정으로 진행한다.

```python
best_k = 7
best_weights = "distance"
best_p = 1
```

정리하면, 현재 교차검증 기준에서는 **K=7, weights="distance", p=1(Manhattan)** 조합이 가장 적절하다. 이제 이 조합으로 전체 학습 데이터에 다시 학습시키고, 테스트 데이터에서 최종 성능을 확인한다.

## 3. 최종 모델 재학습 및 테스트 평가

이제 교차검증에서 가장 좋았던 조합을 가지고 최종 모델을 만든다. 이 단계에서는 학습 데이터를 전부 사용해서 다시 학습하고, 지금까지 모델 선택에 쓰지 않았던 테스트 데이터로 마지막 성능을 확인한다.

In [101]:
from sklearn.metrics import f1_score, precision_score, recall_score

# 교차검증에서 가장 좋았던 최종 후보 조합 설정
best_k = 7
best_weights = "distance"
best_p = 1

# 최종 KNN 모델 생성
final_model = KNeighborsRegressor(
    n_neighbors = best_k,
    weights = best_weights,
    p = best_p,
)

# 전체 학습 데이터로 최종 모델 재학습
final_model.fit(X_train_knn, y_train_knn)

# 테스트 데이터에 대해 구매 확률과 0.5 기준 예측 라벨 생성
test_proba = final_model.predict(X_test_knn)
test_pred = (test_proba >= 0.5).astype(int)

# 최종 테스트 성능 지표 계산
final_metrics = pd.DataFrame([
    {
        "ROC_AUC": roc_auc_score(y_test_knn, test_proba),
        "PR_AUC": average_precision_score(y_test_knn, test_proba),
        "Brier": brier_score_loss(y_test_knn, test_proba),
        "Precision": precision_score(y_test_knn, test_pred, zero_division = 0),
        "Recall": recall_score(y_test_knn, test_pred),
        "F1": f1_score(y_test_knn, test_pred),
    }
])

# 최종 모델 설정과 테스트셋 예측 분포 출력
print("최종 모델")
print(f"K = {best_k}, weights = {best_weights}, p = {best_p}")
print(f"테스트 예측 확률 범위: {test_proba.min():.4f} ~ {test_proba.max():.4f}")
print(f"테스트 평균 예측 확률: {test_proba.mean():.4f}")
print(f"테스트 실제 구매 비율: {y_test_knn.mean():.4f}")

# 최종 테스트 성능표 출력
display(final_metrics)

최종 모델
K = 7, weights = distance, p = 1
테스트 예측 확률 범위: 0.0000 ~ 1.0000
테스트 평균 예측 확률: 0.3429
테스트 실제 구매 비율: 0.1549


,ROC_AUC,PR_AUC,Brier,Precision,Recall,F1
0,0.67479,0.248152,0.227625,0.247045,0.54712,0.340391


## 4. 기준 모델 대비 개선 분석

처음 기준 모델은 `K=5`, `weights="uniform"`, `p=2(Euclidean)`에 가까운 단순한 KNN이었다. 이 모델은 최종 성능을 내기 위한 모델이라기보다, KNN이 예측 확률을 어떻게 만드는지 확인하기 위한 출발점이었다.

GridSearchCV 결과를 기준으로 보면, 최종 후보 모델은 이 기준 설정보다 성능이 좋아졌다.

| 구분 | K | weights | p | 거리 기준 | CV ROC-AUC |
|---|---:|---|---:|---|---:|
| 기준 모델에 가까운 설정 | 5 | uniform | 2 | Euclidean | 0.890562 |
| 최종 후보 모델 | 7 | distance | 1 | Manhattan | 0.921990 |

개선 폭은 다음과 같다.

```text
0.921990 - 0.890562 = 0.031428
```

즉, 단순 기준 설정보다 최종 후보 모델이 CV ROC-AUC 기준으로 약 **0.0314p** 높았다.

이 개선은 크게 두 가지에서 온 것으로 볼 수 있다.

1. `weights="distance"`를 사용하면서, 정말 가까운 이웃의 구매 여부를 더 중요하게 반영했다.
2. `p=1`, 즉 Manhattan 거리를 사용하면서, 큰 차이를 제곱해서 키우기보다 변수 차이를 절댓값 기준으로 누적했다.

또 `K=7`이 선택된 점도 의미가 있다. `K=5`보다 이웃을 조금 더 넓게 보면서도, distance 가중치 덕분에 가까운 이웃의 영향은 유지할 수 있었다. 그래서 너무 예민하지도, 너무 둔감하지도 않은 균형점으로 볼 수 있다.

결국 기준 모델은 KNN의 작동 방식을 보기 위한 출발점이었고, 최종 후보 모델은 K값, 거리 기준, 가중치 방식을 조정해서 구매자와 비구매자를 더 잘 구분하도록 개선된 모델이다.

## 5. KNN 결과로 확인할 수 있는 인사이트

KNN은 회귀계수처럼 “이 컬럼이 구매를 몇 퍼센트 올린다”고 바로 말해주는 모델은 아니다. 대신 비슷한 방문자들이 어떻게 묶이고, 그 안에서 실제 구매자가 얼마나 있는지를 보는 방식으로 해석한다.

그래서 여기서는 다음 네 가지를 확인한다.

1. 예측 확률이 높은 구간에서 실제 구매율도 같이 높아지는가
2. 예측 상위 그룹과 하위 그룹의 행동 패턴은 어떻게 다른가
3. 어떤 컬럼을 섞었을 때 모델 성능이 크게 떨어지는가
4. 예측 확률이 높은 방문자의 가까운 이웃들은 실제로 어떤 구매 패턴을 보이는가

이 분석의 핵심은 구매 가능성이 높은 방문자군을 찾고, 그 방문자들이 어떤 행동 특징을 가지는지 보는 것이다.

### 5-1. 예측 확률 구간별 실제 구매율

먼저 예측 확률을 낮은 순서부터 높은 순서까지 10개 구간으로 나눠본다. 모델이 잘 작동한다면, 상위 구간으로 갈수록 실제 구매율도 같이 높아져야 한다.

In [102]:
# 테스트셋의 예측 확률과 실제 구매 여부를 하나의 표로 정리
test_result = pd.DataFrame({
    "predicted_proba": test_proba,
    "actual_revenue": y_test_knn,
})

# score_decile: 예측 확률 순위 기준 10개 구간, 10이 가장 높은 구간
test_result["score_decile"] = pd.qcut(
    test_result["predicted_proba"].rank(method = "first"),
    q = 10,
    labels = range(1, 11),
).astype(int)

# 예측 점수 구간별 실제 구매율과 구매자 수 집계
decile_result = (
    test_result
    .groupby("score_decile")
    .agg(
        count = ("actual_revenue", "size"),
        mean_predicted_proba = ("predicted_proba", "mean"),
        actual_purchase_rate = ("actual_revenue", "mean"),
        actual_buyers = ("actual_revenue", "sum"),
    )
    .reset_index()
)

# 상위 구간으로 갈수록 실제 구매율이 높아지는지 확인
display(decile_result)

,score_decile,count,mean_predicted_proba,actual_purchase_rate,actual_buyers
0,1,247,0.000000,0.056680,14
1,2,247,0.000000,0.056680,14
2,3,246,0.000000,0.036585,9
3,4,247,0.052480,0.121457,30
4,5,246,0.190389,0.191057,47
5,6,247,0.338921,0.137652,34
6,7,246,0.479984,0.182927,45
7,8,247,0.633523,0.222672,55
8,9,246,0.785184,0.256098,63
9,10,247,0.949343,0.287449,71


### 5-2. 예측 상위 그룹과 하위 그룹의 행동 차이

이번에는 예측 확률 상위 10%와 하위 10%를 비교한다. 입력값은 이미 `log1p`와 `StandardScaler`가 적용된 상태라서 원래 단위 그대로 해석하면 안 된다. 대신 값이 클수록 학습 데이터 평균보다 상대적으로 높은 편이라고 보면 된다.

In [103]:
# 상위 10%, 하위 10%를 추출하기 위한 boolean mask 생성
top_group = test_result["score_decile"] == 10
bottom_group = test_result["score_decile"] == 1

# 상위/하위 그룹의 표준화된 입력 변수 평균 계산
top_profile = X_test_knn.loc[top_group].mean()
bottom_profile = X_test_knn.loc[bottom_group].mean()

# 두 그룹의 평균 차이를 하나의 표로 정리
profile_compare = pd.DataFrame({
    "top_10pct_mean": top_profile,
    "bottom_10pct_mean": bottom_profile,
})
profile_compare["difference"] = profile_compare["top_10pct_mean"] - profile_compare["bottom_10pct_mean"]
profile_compare["abs_difference"] = profile_compare["difference"].abs()

# 차이가 큰 변수부터 확인할 수 있도록 정렬
profile_compare = profile_compare.sort_values("abs_difference", ascending = False)

# 예측 상위 그룹과 하위 그룹의 행동 차이 출력
display(profile_compare)

,top_10pct_mean,bottom_10pct_mean,difference,abs_difference
ProductRelated,0.581198,-0.547101,1.128300,1.128300
ExitRates,-0.500642,0.617087,-1.117729,1.117729
ProductRelated_Duration,0.510282,-0.504557,1.014838,1.014838
Administrative_Duration,0.447333,-0.556864,1.004197,1.004197
Administrative,0.427811,-0.524853,0.952664,0.952664
Informational_Duration,0.548483,-0.256203,0.804687,0.804687
BounceRates,-0.368039,0.432877,-0.800916,0.800916
Informational,0.513276,-0.276610,0.789886,0.789886
is_new_visitor,0.297948,-0.256201,0.554149,0.554149
SpecialDay,-0.205544,0.232482,-0.438026,0.438026


### 5-3. Permutation Importance

KNN에는 Random Forest처럼 기본 feature importance가 없다. 그래서 각 컬럼을 하나씩 섞어보고, 그때 ROC-AUC가 얼마나 떨어지는지 확인한다.

어떤 컬럼을 섞었을 때 성능이 많이 떨어진다면, 모델이 그 컬럼의 정보를 꽤 중요하게 사용했다는 뜻이다. 다만 이것은 인과관계가 아니라, 어디까지나 모델이 의존한 예측 신호로 해석한다.

In [104]:
from sklearn.inspection import permutation_importance

# permutation importance도 ROC-AUC 감소량 기준으로 계산
permutation_scorer = make_scorer(roc_auc_score, response_method = "predict")

# 각 컬럼을 반복적으로 섞어 모델 성능 감소량 측정
importance = permutation_importance(
    final_model,
    X_test_knn,
    y_test_knn,
    scoring = permutation_scorer,
    n_repeats = 5,
    random_state = 42,
)

# 컬럼별 중요도 평균과 표준편차를 표로 정리
importance_result = pd.DataFrame({
    "feature": X_test_knn.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
})

# 성능 감소가 큰 변수부터 확인할 수 있도록 정렬
importance_result = importance_result.sort_values("importance_mean", ascending = False)

# KNN 모델이 많이 의존한 예측 신호 출력
display(importance_result)

,feature,importance_mean,importance_std
5,ProductRelated_Duration,0.037106,0.006407
7,ExitRates,0.033379,0.004082
4,ProductRelated,0.022056,0.011088
1,Administrative_Duration,0.020799,0.006661
10,is_new_visitor,0.015081,0.008072
8,SpecialDay,0.012527,0.004589
9,Weekend,0.008702,0.008004
6,BounceRates,0.007147,0.010189
0,Administrative,0.005733,0.008588
3,Informational_Duration,0.005423,0.005336


### 5-4. KNN 이웃 예시 확인

마지막으로 KNN답게 실제 이웃을 하나 확인해본다. 예측 확률이 가장 높은 방문자 1명을 고르고, 그 방문자와 가까운 이웃 7명 중 실제 구매자가 몇 명인지 본다.

KNN의 예측은 결국 가까운 이웃들의 정답에서 나오기 때문에, 이 예시는 모델이 왜 높은 점수를 줬는지 설명할 때 가장 직관적으로 사용할 수 있다.

In [105]:
# 예측 확률이 가장 높은 테스트 방문자 1명을 선택
target_idx = int(test_result.sort_values("predicted_proba", ascending = False).index[0])

# 선택한 방문자와 가장 가까운 학습 데이터 이웃을 찾음
distances, neighbor_idx = final_model.kneighbors(X_test_knn.iloc[[target_idx]])

# 가까운 이웃들의 거리와 실제 구매 여부를 표로 정리
neighbor_result = pd.DataFrame({
    "neighbor_rank": range(1, best_k + 1),
    "distance": distances[0],
    "neighbor_revenue": y_train_knn.iloc[neighbor_idx[0]].to_numpy(),
})

# 대상 방문자의 예측값, 실제값, 이웃 구매자 수 출력
print(f"대상 test index: {target_idx}")
print(f"예측 구매 확률: {test_result.loc[target_idx, 'predicted_proba']:.4f}")
print(f"실제 구매 여부: {test_result.loc[target_idx, 'actual_revenue']}")
print(f"가까운 {best_k}명 중 구매자 수: {neighbor_result['neighbor_revenue'].sum()}명")

# 가까운 이웃들의 상세 정보 출력
display(neighbor_result)

대상 test index: 1008
예측 구매 확률: 1.0000
실제 구매 여부: 0
가까운 7명 중 구매자 수: 7명


,neighbor_rank,distance,neighbor_revenue
0,1,1.188398,1
1,2,1.247638,1
2,3,1.297301,1
3,4,1.355055,1
4,5,1.405417,1
5,6,1.409166,1
6,7,1.416299,1


### 5-5. 최종 해석

1. **방문 행동 안에 구매 가능성을 구분할 수 있는 신호가 있었다.**  
   상품 페이지 탐색량, 체류 시간, 이탈률, 신규 방문자 여부 같은 행동 정보만으로도 구매 가능성 순위를 만들 수 있었다.

2. **가까운 이웃을 더 중요하게 보는 방식이 효과적이었다.**  
   `uniform`보다 `distance` 가중치가 더 좋은 결과를 보였다는 점은, 단순히 주변 이웃 전체를 똑같이 평균내는 것보다 정말 비슷한 방문자의 구매 여부를 더 크게 보는 것이 낫다는 뜻이다.

3. **Manhattan 거리가 Euclidean 거리보다 조금 더 잘 맞았다.**  
   이 데이터에는 페이지 수나 체류 시간처럼 분포가 한쪽으로 긴 변수가 많다. 그래서 큰 차이를 제곱해서 반영하는 Euclidean보다, 절댓값 차이를 합산하는 Manhattan 방식이 조금 더 안정적으로 작동한 것으로 볼 수 있다.

4. **K=7은 세부 패턴과 안정성 사이의 균형점이었다.**  
   너무 작은 K는 노이즈에 민감하고, 너무 큰 K는 개별 방문자의 특징을 흐릴 수 있다. 이번 탐색에서는 가까운 7명을 보되, 거리 가중치로 더 가까운 이웃을 강하게 반영하는 방식이 가장 좋았다.

5. **이 모델은 구매 가능성이 높은 방문자군을 찾는 기준선으로 쓸 수 있다.**  
   KNN은 방문자 행동 패턴을 바탕으로 구매 가능성이 높은 세션을 우선순위화하고, 이후 고객 세그먼트 분류, 개인화 추천, 리타게팅 우선순위 설정 같은 의사결정 기준으로 확장할 수 있다.

결론적으로, KNN 모델은 온라인 쇼핑 방문자의 행동 패턴만으로도 구매 가능성이 높은 방문자를 어느 정도 구분할 수 있었다. 특히 가까운 이웃을 더 중요하게 반영하는 `Manhattan + distance` 방식이 가장 좋은 결과를 보였고, 이 모델은 구매 가능성이 높은 고객군을 선별하는 타겟팅 기준선으로 활용할 수 있다.